# Recoverable 16,000-Step Single-Formula Symbolic Regression: Iteration 1

## TL;DR

This notebook is a minimally changed replacement for the stalled version-06 shape search. It preserves the original mathematical structure, 27 shape inputs, operators, complexity limits, 50,000-row evolutionary batches and the total `2,000 x 8 = 16,000` population-iteration target.

The final result is still one shape formula:

`stress = case_mean + exp(case_log_scale) * single_shape_formula`

Only execution control is changed. The search is divided into 20 sequential warm-start segments. Each segment runs in an isolated process, writes a checkpoint and is watched for file activity. A stalled worker is stopped and retried from its checkpoint, while all previously completed segments remain saved.

## Context and Scope

### What remains unchanged

- The completed Iteration-1 case-mean and case-log-scale formulas are reused from version 06.
- Every one of the 119 training cases contributes the same locked 5,000-row discovery design, for 595,000 discovery rows.
- Shape evolution uses all 595,000 rows as the available training pool and a 50,000-row PySR batch per fitness cycle.
- The single shape formula uses the same 27 local, within-case-standardised and case-context predictors.
- The operator set remains `+`, `-`, `*`, `/`, `square`, `cube` and `abs`; `maxsize=28` and `maxdepth=10`.
- Formula selection reads all elements in all 15 validation cases. The 15 internal-test cases are evaluated once after selection. The 50 final-test cases remain sealed.

### What changes to prevent a freeze

- One 16,000-progress process is replaced by 20 warm-start segments of 800 population iterations each. Cumulative planned progress is written to `search_progress.json`.
- Julia uses multithreading inside one isolated worker instead of a persistent multiprocessing worker pool.
- The parent controller stops a segment after 45 minutes without file activity or after three hours of wall time, then returns to the last stable segment checkpoint.
- Every successful segment stores a separate stable checkpoint snapshot, so a checkpoint interrupted during writing is not trusted.
- Worker memory is released when each segment exits. Rerunning this notebook skips completed segments.

Only successfully completed segments count toward 16,000. A failed attempt can add wasted wall time, but it is excluded from the accepted cumulative search state and recorded in the attempt audit.

## 1. Fresh Kernel and Setup

Run this notebook in a fresh NotebookCT3 kernel. Stop any older PySR notebook first so its Julia processes do not compete for CPU or memory. On macOS, the controller starts `caffeinate` while the run is active, but the laptop lid should still remain open and power connected.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_package_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'recoverable_single_shape_symbolic.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the NotebookCT3 package root.')


PACKAGE_ROOT = resolve_package_root()
if str(PACKAGE_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT / 'src'))

from recoverable_single_shape_symbolic import (
    RecoverableSingleShapeConfig,
    package_output_dir,
    preflight_recoverable_iteration,
    run_recoverable_iteration_one,
)

print('Package root:', PACKAGE_ROOT)
print('Python:', sys.executable)

## 2. Locked Search Configuration

`RUN_RECOVERABLE_SEARCH=True` starts or resumes the search. Do not change `output_subdir` between retries. A deliberate change to features or search settings must use a new output subdirectory so incompatible checkpoints cannot be mixed.

Engineering acceptance gates are still calculated and exported as diagnostics, but they are not hard rejection rules in this time-limited prototype. Selection first keeps candidates within 3% of the best validation macro RMSE and then prefers the simpler formula, using validation P99 error as a tie-break.

In [ ]:
RUN_RECOVERABLE_SEARCH = True

CONFIG = RecoverableSingleShapeConfig(
    iteration=1,
    output_subdir='iteration_1',
    rows_per_training_case=5_000,
    total_niterations=2_000,
    populations=8,
    segment_niterations=100,
    population_size=40,
    ncycles_per_iteration=100,
    batch_size=50_000,
    maxsize=28,
    maxdepth=10,
    julia_threads=8,
    no_activity_timeout_seconds=45 * 60,
    segment_wall_timeout_seconds=3 * 60 * 60,
    watchdog_poll_seconds=60,
    max_attempts_per_segment=3,
    max_shape_candidates_for_full_validation=14,
    include_old_partial_frontier=True,
    force_rebuild_training_cache=False,
)

OUTPUT_DIR = package_output_dir(PACKAGE_ROOT, CONFIG)
print('Output directory:', OUTPUT_DIR)
print('Segments:', CONFIG.n_segments)
print('Population iterations per segment:', CONFIG.population_iterations_per_segment)
print('Planned cumulative population iterations:', CONFIG.target_population_iterations)
display(pd.DataFrame([CONFIG.__dict__]).T.rename(columns={0: 'value'}))

## 3. Preflight

This check confirms the frozen 119/15/15 development split, 50 sealed final-test cases, 27 shape features, completed case-level formulas, worker script and exact 16,000 planned search target. It does not read final-test element data.

In [ ]:
PREFLIGHT = preflight_recoverable_iteration(PACKAGE_ROOT, CONFIG)
display(PREFLIGHT['checks'])

print('Train cases:', len(PREFLIGHT['inputs']['train_ids']))
print('Validation cases:', len(PREFLIGHT['inputs']['validation_ids']))
print('Internal-test cases:', len(PREFLIGHT['inputs']['internal_ids']))
print('Sealed final-test cases:', len(PREFLIGHT['inputs']['final_ids']))

## 4. Run or Resume

Run this cell once and leave it active. It prints a short watchdog message about every five minutes. Detailed PySR logs are stored under `segments/segment_XX/attempt_Y/`.

If the kernel, VS Code or computer is interrupted, rerun the notebook from the top. Completed `complete.json` markers are skipped and the unfinished segment resumes from the shared checkpoint. Do not manually delete the shared checkpoint while keeping segment markers.

In [ ]:
RESULT = None
if RUN_RECOVERABLE_SEARCH:
    RESULT = run_recoverable_iteration_one(PACKAGE_ROOT, CONFIG)
    display(pd.DataFrame([RESULT]))
else:
    print('Search skipped because RUN_RECOVERABLE_SEARCH=False.')

## 5. Progress and Saved Results

During the search, `search_progress.json` is the clearest cumulative indicator. Each successfully completed segment adds 800 planned population iterations. `segment_attempt_audit.csv` records stalls, timeouts, retries and checkpoint recovery.

After all 20 segments, the notebook validates shortlisted formulas on complete cases and exports one combined symbolic formula. P95/P99, hotspot and engineering-gate fields remain available for interpretation even though they are not mandatory acceptance filters in this prototype.

In [ ]:
progress_path = OUTPUT_DIR / 'search_progress.json'
if progress_path.exists():
    display(pd.DataFrame([json.loads(progress_path.read_text(encoding='utf-8'))]))

attempt_path = OUTPUT_DIR / 'segment_attempt_audit.csv'
if attempt_path.exists():
    display(pd.read_csv(attempt_path).tail(20))

artifacts = {
    'completion': OUTPUT_DIR / 'round_complete.json',
    'formula': OUTPUT_DIR / 'selected_composite_formula.csv',
    'formula_text': OUTPUT_DIR / 'selected_composite_formula.txt',
    'split_metrics': OUTPUT_DIR / 'selected_formula_split_metrics.csv',
    'candidate_metrics': OUTPUT_DIR / 'shape_candidate_validation_metrics.csv',
}
display(pd.DataFrame([
    {'artifact': name, 'exists': path.exists(), 'path': str(path)}
    for name, path in artifacts.items()
]))

if artifacts['completion'].exists():
    print(artifacts['formula_text'].read_text(encoding='utf-8'))
    display(pd.read_csv(artifacts['split_metrics']))
    candidates = pd.read_csv(artifacts['candidate_metrics'])
    display(candidates.sort_values(
        ['selected_candidate', 'validation_macro_rmse'],
        ascending=[False, True],
    ).head(20))
else:
    print('The run is not complete. The same notebook can be rerun safely.')

## Takeaways and Interpretation Boundary

This notebook tests whether the original single-shape symbolic design can complete reliably under the available time and hardware. The selected equation is an Iteration-1 prototype, not yet an engineering-qualified lifetime model. It predicts FEM maximum-principal stress when post-deformation coordinates and the three parameter fields are available.

Lifetime conversion still requires a professor-confirmed strength, damage or failure relationship. The sealed 50-case final test must not be used until formula structure and constants have been fixed across the development workflow.